# RNN and LSTM Models for Cat vs Dog Classification

This notebook implements:
- **(a)** Customized RNN and LSTM architectures for image classification
- **(b)** Variants of RNN and LSTM with performance comparison
- **(c)** Analysis and comparison of results from RNN and LSTM models
- **(d)** Graphs to visualize loss and accuracy metrics

## 1. Install and Import Dependencies

In [ ]:
!pip install kagglehub torch torchvision matplotlib scikit-learn seaborn pandas numpy --quiet

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Download and Prepare Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print("Path to dataset files:", path)

In [ ]:
# Explore the dataset directory structure
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        sub_indent = ' ' * 2 * (level + 1)
        for f in files[:5]:
            print(f"{sub_indent}{f}")
        if len(files) > 5:
            print(f"{sub_indent}... and {len(files) - 5} more files")

In [ ]:
# Collect image paths and labels
# The dataset typically has 'dogs' and 'cats' folders, or images prefixed with labels
image_paths = []
labels = []

# Walk through all subdirectories to find images
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            full_path = os.path.join(root, f)
            # Determine label from filename or parent folder
            fname_lower = f.lower()
            parent_lower = os.path.basename(root).lower()
            
            if 'cat' in fname_lower or 'cat' in parent_lower:
                image_paths.append(full_path)
                labels.append(0)  # 0 = Cat
            elif 'dog' in fname_lower or 'dog' in parent_lower:
                image_paths.append(full_path)
                labels.append(1)  # 1 = Dog

print(f"Total images found: {len(image_paths)}")
print(f"Cats: {labels.count(0)}, Dogs: {labels.count(1)}")

In [ ]:
# Take a small sample for feasible training with RNN/LSTM
SAMPLE_SIZE = 2000  # 1000 cats + 1000 dogs (or total 2000 balanced)

# Separate by class and sample equally
cat_indices = [i for i, l in enumerate(labels) if l == 0]
dog_indices = [i for i, l in enumerate(labels) if l == 1]

n_per_class = min(SAMPLE_SIZE // 2, len(cat_indices), len(dog_indices))
print(f"Sampling {n_per_class} images per class ({2 * n_per_class} total)")

random.shuffle(cat_indices)
random.shuffle(dog_indices)

sampled_indices = cat_indices[:n_per_class] + dog_indices[:n_per_class]
random.shuffle(sampled_indices)

sampled_paths = [image_paths[i] for i in sampled_indices]
sampled_labels = [labels[i] for i in sampled_indices]

print(f"Sampled - Cats: {sampled_labels.count(0)}, Dogs: {sampled_labels.count(1)}")

In [ ]:
# Visualize some sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images from Dataset', fontsize=16)

class_names = ['Cat', 'Dog']
for i, ax in enumerate(axes.flat):
    img = Image.open(sampled_paths[i]).convert('RGB')
    ax.imshow(img)
    ax.set_title(class_names[sampled_labels[i]], fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Create Dataset and DataLoaders

**Key Idea:** To use RNN/LSTM for image classification, we treat each image row (scanline) as a time step in the sequence. For a 64×64 image, we get a sequence of 64 time steps, each with 64×3 features (width × channels).

In [ ]:
IMG_SIZE = 64  # Resize images to 64x64

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class CatDogDataset(Dataset):
    """Custom Dataset for Cat vs Dog images."""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        # Reshape: (C, H, W) -> (H, W*C) to treat rows as sequence steps
        # img shape: (3, 64, 64) -> (64, 192)
        img = img.permute(1, 2, 0)       # (H, W, C)
        img = img.reshape(IMG_SIZE, -1)   # (H, W*C) = (64, 192)
        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
# Train/Validation/Test split: 70% / 15% / 15%
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    sampled_paths, sampled_labels, test_size=0.3, random_state=SEED, stratify=sampled_labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Create datasets
train_dataset = CatDogDataset(X_train, y_train, transform)
val_dataset   = CatDogDataset(X_val, y_val, transform)
test_dataset  = CatDogDataset(X_test, y_test, transform)

# Create dataloaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verify shapes
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")   # (batch, seq_len=64, input_size=192)
print(f"Labels shape: {sample_labels.shape}")

## 4. (a) Customized RNN and LSTM Architectures

In [ ]:
# =====================================================
# Model 1: Vanilla RNN
# =====================================================
class VanillaRNN(nn.Module):
    """Basic RNN for image classification.
    Treats each row of the image as a time step."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)       # out: (batch, seq_len, hidden_size)
        out = out[:, -1, :]            # Take last time step
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✓ VanillaRNN defined")

In [ ]:
# =====================================================
# Model 2: Bidirectional RNN
# =====================================================
class BiRNN(nn.Module):
    """Bidirectional RNN variant - processes sequence in both directions."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(BiRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = out[:, -1, :]       # Last time step (concatenated both directions)
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✓ BiRNN defined")

In [ ]:
# =====================================================
# Model 3: Vanilla LSTM
# =====================================================
class VanillaLSTM(nn.Module):
    """Standard LSTM for image classification."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(VanillaLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]            # Take last time step
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✓ VanillaLSTM defined")

In [ ]:
# =====================================================
# Model 4: Bidirectional LSTM
# =====================================================
class BiLSTM(nn.Module):
    """Bidirectional LSTM - processes sequence in both directions."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✓ BiLSTM defined")

In [ ]:
# =====================================================
# Model 5: Stacked LSTM with Attention
# =====================================================
class LSTMWithAttention(nn.Module):
    """LSTM with a simple self-attention mechanism over time steps."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        # Attention layer
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        lstm_out, _ = self.lstm(x, (h0, c0))  # (batch, seq_len, hidden)
        
        # Attention weights
        attn_weights = self.attention(lstm_out)          # (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        
        # Weighted sum of LSTM outputs
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (batch, hidden)
        
        out = self.dropout(context)
        out = self.fc(out)
        return out

print("✓ LSTMWithAttention defined")

In [ ]:
# =====================================================
# Model 6: GRU (Gated Recurrent Unit - RNN variant)
# =====================================================
class GRUModel(nn.Module):
    """GRU - a popular variant of RNN with gating mechanisms."""
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("✓ GRUModel defined")

## 5. Model Summary
Display architecture details and parameter counts for all models.

In [ ]:
INPUT_SIZE = IMG_SIZE * 3   # 64 * 3 = 192 (width * channels)
SEQ_LEN    = IMG_SIZE       # 64 (height = number of time steps)
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
NUM_CLASSES = 2
DROPOUT     = 0.3

# Instantiate all models
models_dict = {
    'Vanilla RNN':       VanillaRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
    'Bidirectional RNN': BiRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
    'Vanilla LSTM':      VanillaLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
    'Bidirectional LSTM':BiLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
    'LSTM + Attention':  LSTMWithAttention(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
    'GRU':               GRUModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT),
}

# Print parameter counts
print(f"{'Model':<25} {'Parameters':>12}")
print('=' * 40)
for name, model in models_dict.items():
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{name:<25} {total_params:>12,}")

## 6. (b) Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_targets

print("✓ Training and evaluation functions defined")

In [ ]:
def train_model(model, model_name, train_loader, val_loader, device,
                num_epochs=20, lr=0.001):
    """Full training loop with history tracking."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                     patience=3, factor=0.5)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_val_acc = 0.0
    
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader,
                                                 criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:>2}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    return history

print("✓ Training pipeline defined")

## 7. Train All Models

In [ ]:
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

# Store all histories and trained models
all_histories = {}
trained_models = {}

for model_name, model in models_dict.items():
    history = train_model(
        model, model_name,
        train_loader, val_loader, device,
        num_epochs=NUM_EPOCHS, lr=LEARNING_RATE
    )
    all_histories[model_name] = history
    trained_models[model_name] = model

print("\n" + "="*60)
print("All models trained successfully!")
print("="*60)

## 8. (c) Analyze and Compare Results

In [ ]:
# Evaluate all models on the test set
criterion = nn.CrossEntropyLoss()
test_results = {}

print(f"\n{'Model':<25} {'Test Loss':>10} {'Test Acc':>10}")
print('=' * 48)

for model_name, model in trained_models.items():
    test_loss, test_acc, preds, targets = evaluate(model, test_loader, criterion, device)
    test_results[model_name] = {
        'test_loss': test_loss,
        'test_acc': test_acc,
        'predictions': preds,
        'targets': targets
    }
    print(f"{model_name:<25} {test_loss:>10.4f} {test_acc:>9.2f}%")

In [ ]:
# Detailed classification report for each model
class_names = ['Cat', 'Dog']

for model_name, result in test_results.items():
    print(f"\n{'='*60}")
    print(f"Classification Report: {model_name}")
    print('='*60)
    print(classification_report(result['targets'], result['predictions'],
                                target_names=class_names))

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Confusion Matrices for All Models', fontsize=16, fontweight='bold')

for idx, (model_name, result) in enumerate(test_results.items()):
    ax = axes[idx // 3, idx % 3]
    cm = confusion_matrix(result['targets'], result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{model_name}\nAcc: {result['test_acc']:.2f}%", fontsize=12)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
summary_data = []
for model_name in test_results:
    summary_data.append({
        'Model': model_name,
        'Type': 'RNN' if 'RNN' in model_name or 'GRU' in model_name else 'LSTM',
        'Best Train Acc (%)': max(all_histories[model_name]['train_acc']),
        'Best Val Acc (%)': max(all_histories[model_name]['val_acc']),
        'Test Acc (%)': test_results[model_name]['test_acc'],
        'Final Train Loss': all_histories[model_name]['train_loss'][-1],
        'Final Val Loss': all_histories[model_name]['val_loss'][-1],
        'Test Loss': test_results[model_name]['test_loss'],
        'Parameters': sum(p.numel() for p in trained_models[model_name].parameters())
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Test Acc (%)', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("OVERALL MODEL COMPARISON (Ranked by Test Accuracy)")
print("="*80)
print(summary_df.to_string(index=False))

## 9. (d) Visualization - Loss and Accuracy Graphs

In [ ]:
# Plot 1: Individual Training Curves for Each Model
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Training & Validation Curves for Each Model', fontsize=16, fontweight='bold')

for idx, (model_name, history) in enumerate(all_histories.items()):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Plot loss
    ax.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, history['val_loss'], 'r--', label='Val Loss', linewidth=2)
    
    # Plot accuracy on secondary y-axis
    ax2 = ax.twinx()
    ax2.plot(epochs, history['train_acc'], 'g-', label='Train Acc', linewidth=2, alpha=0.7)
    ax2.plot(epochs, history['val_acc'], 'm--', label='Val Acc', linewidth=2, alpha=0.7)
    
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='blue')
    ax2.set_ylabel('Accuracy (%)', color='green')
    
    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Comparative Training Loss across all models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Training Loss comparison
ax1 = axes[0]
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], color=colors[idx],
             label=model_name, linewidth=2, marker='o', markersize=3)

ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Validation Loss comparison
ax2 = axes[1]
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs = range(1, len(history['val_loss']) + 1)
    ax2.plot(epochs, history['val_loss'], color=colors[idx],
             label=model_name, linewidth=2, marker='s', markersize=3)

ax2.set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Comparative Accuracy across all models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training Accuracy comparison
ax1 = axes[0]
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs = range(1, len(history['train_acc']) + 1)
    ax1.plot(epochs, history['train_acc'], color=colors[idx],
             label=model_name, linewidth=2, marker='o', markersize=3)

ax1.set_title('Training Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Validation Accuracy comparison
ax2 = axes[1]
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs = range(1, len(history['val_acc']) + 1)
    ax2.plot(epochs, history['val_acc'], color=colors[idx],
             label=model_name, linewidth=2, marker='s', markersize=3)

ax2.set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Test Accuracy Bar Chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

model_names = list(test_results.keys())
test_accs = [test_results[m]['test_acc'] for m in model_names]
test_losses = [test_results[m]['test_loss'] for m in model_names]

# Test Accuracy bar chart
bars1 = axes[0].bar(range(len(model_names)), test_accs, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([min(test_accs) - 5, max(test_accs) + 5])
for bar, acc in zip(bars1, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Test Loss bar chart
bars2 = axes[1].bar(range(len(model_names)), test_losses, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Test Loss Comparison', fontsize=14, fontweight='bold')
for bar, loss in zip(bars2, test_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{loss:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 5: RNN Family vs LSTM Family grouped comparison
fig, ax = plt.subplots(figsize=(12, 6))

rnn_models = ['Vanilla RNN', 'Bidirectional RNN', 'GRU']
lstm_models = ['Vanilla LSTM', 'Bidirectional LSTM', 'LSTM + Attention']

rnn_accs = [test_results[m]['test_acc'] for m in rnn_models]
lstm_accs = [test_results[m]['test_acc'] for m in lstm_models]

x = np.arange(3)
width = 0.35

bars_rnn = ax.bar(x - width/2, rnn_accs, width, label='RNN Family',
                   color='#3498db', edgecolor='black', linewidth=0.5)
bars_lstm = ax.bar(x + width/2, lstm_accs, width, label='LSTM Family',
                    color='#e74c3c', edgecolor='black', linewidth=0.5)

ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('RNN Family vs LSTM Family - Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Vanilla', 'Bidirectional', 'Advanced\n(GRU / Attention)'], fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars_rnn, bars_lstm]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                f'{height:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 6: Parameter Efficiency - Accuracy vs Number of Parameters
fig, ax = plt.subplots(figsize=(10, 6))

for idx, (model_name, result) in enumerate(test_results.items()):
    params = sum(p.numel() for p in trained_models[model_name].parameters())
    ax.scatter(params, result['test_acc'], s=200, c=colors[idx],
               edgecolors='black', linewidth=1, zorder=5)
    ax.annotate(model_name, (params, result['test_acc']),
                textcoords="offset points", xytext=(10, 5), fontsize=10)

ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Parameter Efficiency: Accuracy vs Model Size', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Analysis and Conclusions

In [ ]:
# Final analysis
print("="*70)
print("ANALYSIS AND CONCLUSIONS")
print("="*70)

# Find best models
best_model = max(test_results, key=lambda x: test_results[x]['test_acc'])
best_rnn = max([m for m in test_results if 'RNN' in m or 'GRU' in m],
               key=lambda x: test_results[x]['test_acc'])
best_lstm = max([m for m in test_results if 'LSTM' in m],
                key=lambda x: test_results[x]['test_acc'])

print(f"\n1. BEST OVERALL MODEL: {best_model} ({test_results[best_model]['test_acc']:.2f}%)")
print(f"   Best RNN variant:  {best_rnn} ({test_results[best_rnn]['test_acc']:.2f}%)")
print(f"   Best LSTM variant: {best_lstm} ({test_results[best_lstm]['test_acc']:.2f}%)")

# RNN vs LSTM comparison
rnn_avg = np.mean([test_results[m]['test_acc'] for m in rnn_models])
lstm_avg = np.mean([test_results[m]['test_acc'] for m in lstm_models])

print(f"\n2. RNN vs LSTM FAMILY COMPARISON:")
print(f"   Average RNN family accuracy:  {rnn_avg:.2f}%")
print(f"   Average LSTM family accuracy: {lstm_avg:.2f}%")
if lstm_avg > rnn_avg:
    print(f"   → LSTM family outperforms RNN family by {lstm_avg - rnn_avg:.2f}%")
else:
    print(f"   → RNN family outperforms LSTM family by {rnn_avg - lstm_avg:.2f}%")

print(f"\n3. EFFECT OF BIDIRECTIONAL PROCESSING:")
rnn_diff = test_results['Bidirectional RNN']['test_acc'] - test_results['Vanilla RNN']['test_acc']
lstm_diff = test_results['Bidirectional LSTM']['test_acc'] - test_results['Vanilla LSTM']['test_acc']
print(f"   Bidirectional vs Vanilla RNN:  {'+' if rnn_diff >=0 else ''}{rnn_diff:.2f}%")
print(f"   Bidirectional vs Vanilla LSTM: {'+' if lstm_diff >= 0 else ''}{lstm_diff:.2f}%")

print(f"\n4. ATTENTION MECHANISM IMPACT:")
attn_diff = test_results['LSTM + Attention']['test_acc'] - test_results['Vanilla LSTM']['test_acc']
print(f"   LSTM+Attention vs Vanilla LSTM: {'+' if attn_diff >=0 else ''}{attn_diff:.2f}%")

print(f"\n5. GRU vs VANILLA RNN:")
gru_diff = test_results['GRU']['test_acc'] - test_results['Vanilla RNN']['test_acc']
print(f"   GRU vs Vanilla RNN: {'+' if gru_diff>=0 else ''}{gru_diff:.2f}%")

print(f"\n6. KEY OBSERVATIONS:")
print(f"   • LSTM models generally handle longer sequences better due to the")
print(f"     gating mechanism that mitigates the vanishing gradient problem.")
print(f"   • Bidirectional variants capture both forward and backward context")
print(f"     from image rows, which can improve spatial understanding.")
print(f"   • The attention mechanism allows the model to focus on the most")
print(f"     informative rows of the image rather than relying solely on the")
print(f"     last hidden state.")
print(f"   • GRU provides a good balance between RNN simplicity and LSTM")
print(f"     performance with fewer parameters.")
print(f"   • Note: CNNs are inherently better suited for image classification;")
print(f"     RNN/LSTM models treat images as sequences which is suboptimal")
print(f"     for spatial feature extraction.")

In [ ]:
# Display final summary table
print("\nFinal Summary Table:")
print(summary_df.to_string(index=False))